# GridFM DataKit Tutorial

`gridfm-datakit` generates synthetic power-flow (PF) and optimal-power-flow (OPF) datasets to train ML solvers for power grids. This notebook walks through the library step by step, from the grid data structure to the files it writes, so that by the end you can read a datakit config, predict what data it will produce, and check that data.

**What you will learn**

| Part | Topic | Needs Julia? |
|---|---|---|
| 1 | The big picture: what a generated sample is and how the pipeline is ordered | no |
| 2 | The `Network` object: buses, generators, branches, cost curves | no (after loading) |
| 3 | Load scenarios: how realistic demand variations are built | only 3.2 |
| 4 | Topology, generator-cost and admittance perturbations | no |
| 5 | The YAML configuration and how it maps to classes | no |
| 6 | Running a full data generation (PF mode) | yes |
| 7 | Reading and sanity-checking the output files | no |
| 8 | Validation and statistics with the CLI | no |
| 9 | PF mode vs OPF mode | yes |

Each part mixes short explanations with runnable cells, and ends with **Try it yourself** suggestions.

## Prerequisites

- **Python 3.10 or 3.11** (3.12 is not supported on Windows). Packages: `gridfm-datakit`, `networkx`, `matplotlib`, `pandas`, `pyarrow`, `pyyaml`, `ipykernel`.
- **Julia with PowerModels, Ipopt and Memento.** The AC power-flow and OPF solvers run in Julia through `juliacall`. Install once with `gridfm_datakit setup_pm`. The first Julia call compiles packages and can take several minutes.
- **Internet access** the first time a PGLib case is used: it is downloaded and cached (override the cache folder with the `GRIDFM_DATAKIT_CACHE_DIR` environment variable).

Run the notebook top to bottom. Generated files go to `tutorial_output/` next to the notebook.

The cell below imports everything used later and checks that Julia and its packages are available, so problems show up immediately.

In [ ]:
import copy
import subprocess
import sys
from importlib import metadata
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import yaml
from IPython.display import Image, display

pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 4)

OUTPUT_DIR = Path("tutorial_output").resolve()
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python        :", sys.version.split()[0])
print("gridfm-datakit:", metadata.version("gridfm-datakit"))

# Fail early, with a clear message, if the Julia side is not ready.
try:
    from juliacall import Main as jl

    jl.seval("using PowerModels, Ipopt, Memento")
    print("Julia         :", jl.seval("string(VERSION)"))
except Exception as exc:
    raise RuntimeError(
        "Julia / PowerModels is not available. Run `gridfm_datakit setup_pm` first."
    ) from exc

---

## Part 1: The big picture

A **sample** in a datakit dataset is one solved operating point of one grid: loads, generator dispatch, voltages, branch flows, plus the grid parameters that produced them. Samples are diversified with four ingredients:

| Ingredient | What varies | Config block | Classes |
|---|---|---|---|
| Load scenarios | Active/reactive demand at every bus | `load` | `LoadScenariosFromAggProfile`, `Powergraph` |
| Topology | N-k outages of branches and generators | `topology_perturbation` | `NMinusKGenerator`, `RandomComponentDropGenerator`, `NoPerturbationGenerator` |
| Generator costs | Cost coefficients (changes the OPF dispatch) | `generation_perturbation` | `PermuteGenCostGenerator`, `PerturbGenCostGenerator`, `NoGenPerturbationGenerator` |
| Admittances | Branch resistance `R` and reactance `X` | `admittance_perturbation` | `PerturbAdmittanceGenerator`, `NoAdmittancePerturbationGenerator` |

The `settings.mode` option decides **when** each ingredient is applied, and therefore what kind of data you get:

```text
For every load scenario k (all loads of the grid set to scenario k):

  mode "pf"  (power flow, may violate operating limits)
    1. cost perturbation
    2. admittance perturbation
    3. AC-OPF on the intact grid            -> generator set-points
    4. topology perturbation (N-0 ... N-k)  -> several variants
    5. AC-PF on each variant, set-points kept (+ DC-PF)

  mode "opf" (all operating limits satisfied)
    1. topology perturbation                -> several variants
    2. cost perturbation
    3. admittance perturbation
    4. AC-OPF on each variant (+ DC-OPF)
```

In `pf` mode the set-points are computed **before** outages, so an outage can push voltages or flows past their limits. That is intentional: it trains PF solvers on stressed states. In `opf` mode every sample is cost-optimal and feasible.

One load scenario therefore produces **several samples** (one per topology variant). Remember this for Part 7: `load_scenario_idx` identifies the load scenario, `scenario` identifies the sample.

The next cell lists the modules of the package, using their docstrings, to show where each piece lives.

In [ ]:
import ast

import gridfm_datakit

pkg_dir = Path(gridfm_datakit.__file__).parent
rows = []
for path in sorted(pkg_dir.rglob("*.py")):
    if "__pycache__" in path.parts or path.name == "__init__.py":
        continue
    if {"dynamic", "powsybl"} & set(path.relative_to(pkg_dir).parts):
        continue  # optional sub-packages, not covered here
    doc = ast.get_docstring(ast.parse(path.read_text(encoding="utf-8"))) or ""
    rows.append(
        {
            "module": ".".join(path.relative_to(pkg_dir.parent).with_suffix("").parts),
            "summary": doc.strip().splitlines()[0] if doc else "",
        }
    )

with pd.option_context("display.max_colwidth", 110):
    display(pd.DataFrame(rows))

---

## Part 2: The `Network` object

Everything starts from `Network` (`gridfm_datakit.network`), a thin wrapper around the four MATPOWER matrices. They are plain **NumPy arrays**, not DataFrames, and their columns are addressed with named index constants from `gridfm_datakit.utils.idx_*`:

| Attribute | One row per | Columns indexed by | Examples |
|---|---|---|---|
| `buses` | bus | `idx_bus` | `PD`, `QD`, `VM`, `VA`, `BUS_TYPE` |
| `gens` | generator | `idx_gen` | `GEN_BUS`, `PG`, `PMAX`, `QMAX`, `GEN_STATUS` |
| `branches` | line or transformer | `idx_brch` | `F_BUS`, `T_BUS`, `BR_R`, `BR_X`, `RATE_A`, `BR_STATUS` |
| `gencosts` | generator | `idx_cost` | `MODEL`, `NCOST`, `COST` (then the coefficients) |

Conventions to keep in mind:

- Bus ids are remapped to a continuous **0-based index**. `bus_index_mapping` and `reverse_bus_index_mapping` convert to and from the ids of the case file.
- There is exactly one **reference (slack) bus** (`ref_bus_idx`), and the grid must be a single connected component.
- Power is in MW / MVAr, angles in degrees, `R`, `X`, `B` in per unit on `baseMVA`.
- Outages are status flags (`BR_STATUS`, `GEN_STATUS`), not deleted rows. Perturbations work on copies made with `copy_for_perturbation()`.

Two loaders exist: `load_net_from_pglib(name)` downloads a case from [PGLib-OPF](https://github.com/power-grid-lib/pglib-opf) (name **without** the `pglib_opf_` prefix) and caches it, and `load_net_from_file(path)` reads a local MATPOWER `.m` file. Both clean the case with PowerModels, hence the need for Julia.

In [ ]:
from gridfm_datakit.network import Network, load_net_from_pglib

NETWORK_NAME = "case14_ieee"  # small enough to keep every later step fast
net = load_net_from_pglib(NETWORK_NAME)

print(f"type              : {type(net).__name__}")
print(f"baseMVA           : {net.baseMVA}")
print(f"buses matrix      : {net.buses.shape}  (rows = buses)")
print(f"gens matrix       : {net.gens.shape}  (rows = generators)")
print(f"branches matrix   : {net.branches.shape}  (rows = branches)")
print(f"gencosts matrix   : {net.gencosts.shape}")
print(f"reference bus     : index {net.ref_bus_idx} (id in the case file: {net.reverse_bus_index_mapping[net.ref_bus_idx]})")
print(f"single component  : {bool(net.check_single_connected_component())}")
print(f"first id mappings : {list(net.bus_index_mapping.items())[:5]}  (case-file id -> internal index)")

### 2.2 A readable view of the matrices

Raw arrays are hard to read, so the next cell builds one DataFrame per component by naming the columns with the index constants. This is only for display: the library itself keeps working on the arrays.

Things to notice in the output:

- `type` is the bus type: **PQ** (load bus, voltage is unknown), **PV** (generator bus, voltage magnitude is fixed) and **REF** (slack bus, also fixes the angle).
- `Vmin_pu` / `Vmax_pu`, `Pmax_MW`, `Qmax_MVAr` and `rate_a_MVA` are the **operating limits**. An OPF respects them, a PF does not.
- The cost polynomial of generator $g$ is $c_2 P_g^2 + c_1 P_g + c_0$.

In [ ]:
from gridfm_datakit.utils.idx_brch import (
    ANGMAX, ANGMIN, BR_B, BR_R, BR_STATUS, BR_X, F_BUS, RATE_A, SHIFT, T_BUS, TAP,
)
from gridfm_datakit.utils.idx_bus import (
    BASE_KV, BS, BUS_I, BUS_TYPE, GS, PD, PQ, PV, QD, REF, VA, VM, VMAX, VMIN,
)
from gridfm_datakit.utils.idx_cost import COST, NCOST
from gridfm_datakit.utils.idx_gen import (
    GEN_BUS, GEN_STATUS, PG, PMAX, PMIN, QG, QMAX, QMIN, VG,
)


def to_frame(matrix: np.ndarray, columns: dict) -> pd.DataFrame:
    return pd.DataFrame({name: matrix[:, idx] for name, idx in columns.items()})


def cost_frame(network: Network) -> pd.DataFrame:
    n_cost = int(network.gencosts[0, NCOST])
    names = [f"c{n_cost - 1 - i}" for i in range(n_cost)]  # highest order first: c2, c1, c0
    return pd.DataFrame(network.gencosts[:, COST:COST + n_cost], columns=names)


bus_df = to_frame(net.buses, {
    "bus": BUS_I, "type": BUS_TYPE, "Pd_MW": PD, "Qd_MVAr": QD, "Gs_MW": GS, "Bs_MVAr": BS,
    "Vm_pu": VM, "Va_deg": VA, "base_kv": BASE_KV, "Vmax_pu": VMAX, "Vmin_pu": VMIN,
})
bus_df["bus"] = bus_df["bus"].astype(int)
bus_df["type"] = bus_df["type"].astype(int).map({PQ: "PQ", PV: "PV", REF: "REF"})

gen_df = pd.concat([
    to_frame(net.gens, {
        "bus": GEN_BUS, "Pg_MW": PG, "Qg_MVAr": QG, "Pmax_MW": PMAX, "Pmin_MW": PMIN,
        "Qmax_MVAr": QMAX, "Qmin_MVAr": QMIN, "Vg_pu": VG, "status": GEN_STATUS,
    }),
    cost_frame(net),
], axis=1)
gen_df["bus"] = gen_df["bus"].astype(int)

branch_df = to_frame(net.branches, {
    "from_bus": F_BUS, "to_bus": T_BUS, "r_pu": BR_R, "x_pu": BR_X, "b_pu": BR_B,
    "rate_a_MVA": RATE_A, "tap": TAP, "shift_deg": SHIFT, "status": BR_STATUS,
    "angmin_deg": ANGMIN, "angmax_deg": ANGMAX,
})
branch_df[["from_bus", "to_bus"]] = branch_df[["from_bus", "to_bus"]].astype(int)

for title, frame in [("Buses", bus_df), ("Generators (with cost coefficients)", gen_df), ("Branches", branch_df)]:
    print(f"--- {title} ---")
    display(frame)
print("Bus types:", bus_df["type"].value_counts().to_dict())

### 2.3 Questions to ask about any grid

Before generating data, check how much room the grid has. These numbers explain later results: a grid with little generation headroom runs into OPF infeasibility quickly when the load is scaled up, and branches with small `rate_a_MVA` overload first.

In [ ]:
summary = pd.Series({
    "buses": len(bus_df),
    "buses with a load (Pd > 0)": int((bus_df["Pd_MW"] > 0).sum()),
    "generators": len(gen_df),
    "branches": len(branch_df),
    "branches with off-nominal tap (transformers)": int(((branch_df["tap"] != 0) & (branch_df["tap"] != 1)).sum()),
    "total demand (MW)": bus_df["Pd_MW"].sum(),
    "total generation capacity (MW)": gen_df["Pmax_MW"].sum(),
    "capacity / demand": gen_df["Pmax_MW"].sum() / bus_df["Pd_MW"].sum(),
})
display(summary.to_frame("value"))

print("Generators per bus:", gen_df.groupby("bus").size().to_dict())
print("\nGenerators ordered by linear cost c1 (cheapest first):")
display(gen_df.sort_values("c1")[["bus", "Pmax_MW", "c2", "c1", "c0"]])

print("Branch impedance and rating:")
display(branch_df[["r_pu", "x_pu", "rate_a_MVA"]].describe().T)

### 2.4 The grid as a graph

A power grid is a graph: buses are nodes, branches are edges. This is also the view taken by graph neural networks later (see GridFM GraphKit). The plot below colors nodes by bus type, sizes them by load, and sets edge width from the branch rating. Parallel branches would collapse into one edge in this simple `networkx.Graph`.

**Try it yourself**
- Which bus is the slack bus, and is it a high-degree node? (`dict(graph.degree)`)
- Load another PGLib case by changing `NETWORK_NAME` (for example `case24_ieee_rts`) and re-run Part 2.

In [ ]:
TYPE_COLORS = {"PQ": "tab:blue", "PV": "tab:green", "REF": "tab:red"}

graph = nx.Graph()
for row in bus_df.itertuples():
    graph.add_node(row.bus, type=row.type, pd=row.Pd_MW)
for row in branch_df.itertuples():
    graph.add_edge(row.from_bus, row.to_bus, rate=row.rate_a_MVA)

pos = nx.spring_layout(graph, seed=0)
node_colors = [TYPE_COLORS[graph.nodes[n]["type"]] for n in graph.nodes]
node_sizes = [150 + 25 * graph.nodes[n]["pd"] for n in graph.nodes]
rates = np.array([graph.edges[e]["rate"] for e in graph.edges])
widths = 1 + 4 * rates / rates.max() if rates.max() > 0 else 1.5

fig, ax = plt.subplots(figsize=(7, 5))
nx.draw_networkx(
    graph, pos, ax=ax, node_color=node_colors, node_size=node_sizes,
    width=widths, edge_color="gray", font_size=8,
)
ax.legend(handles=[plt.Line2D([], [], marker="o", ls="", color=c, label=t) for t, c in TYPE_COLORS.items()])
ax.set_title(f"{NETWORK_NAME}: node size = load, edge width = rating")
ax.axis("off")
plt.show()

---

## Part 3: Load scenarios

The `load` block produces an array of shape `(n_buses, n_scenarios, 2)` holding `(P, Q)` for every bus and scenario. Two generators are available.

### `agg_load_profile` (default)

It combines a **global trend** taken from a real aggregated load curve with **local noise** per load:

1. Find the largest uniform scaling `u` of the nominal load for which the AC-OPF still converges. The load is raised from `start_scaling_factor` in steps of `step_size`, up to `max_scaling_factor`. This step runs OPF, hence Julia.
2. Set the lower bound $l = (1 - \text{global\_range})\,u$.
3. Min-max scale the aggregated profile (a CSV from `gridfm_datakit/load_profiles/`) to $[l, u]$ to get $\text{ref}^k$ for scenario $k$.
4. Scale each load: $\tilde p_i^k = p_i \cdot \text{ref}^k \cdot \varepsilon_i^k$ with $\varepsilon_i^k \sim \mathcal U(1-\sigma, 1+\sigma)$, and the same for $q$ if `change_reactive_power` is true.

If you ask for fewer scenarios than the profile has points, the profile is **cut**. If you ask for more, it is **interpolated**.

### `powergraph`

Only the global trend, $\tilde p_i^k = p_i \cdot \text{agg}^k / \max_k \text{agg}^k$, with fixed reactive power and no local noise.

### 3.1 The aggregated profile

The cell below plots the raw default profile and the same profile min-max scaled to an **illustrative** interval (the real `u` comes from OPF in 3.2), and lists the profiles that ship with the package.

In [ ]:
from importlib import resources

from gridfm_datakit.perturbations.load_perturbation import LoadScenariosFromAggProfile

profile_dir = resources.files("gridfm_datakit.load_profiles")
print("Available profiles:", sorted(p.name for p in profile_dir.iterdir() if p.name.endswith(".csv")))

agg = pd.read_csv(profile_dir.joinpath("default.csv")).to_numpy().ravel()
u_demo, global_range = 1.3, 0.4  # illustrative values only
l_demo = (1 - global_range) * u_demo
ref_demo = LoadScenariosFromAggProfile.min_max_scale(agg, l_demo, u_demo)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(agg)
axes[0].set(title=f"default.csv ({len(agg)} points)", xlabel="time step", ylabel="aggregated load")
axes[1].plot(ref_demo)
axes[1].axhline(u_demo, ls="--", c="r", label="u")
axes[1].axhline(l_demo, ls="--", c="g", label="l = (1 - global_range) * u")
axes[1].set(title="Scaled to [l, u] (illustrative)", xlabel="time step", ylabel="global scaling factor")
axes[1].legend()
plt.tight_layout()
plt.show()

### 3.2 Generate scenarios for our grid

Now the real generator is called on `net`. It searches for `u` with repeated OPF solves (a log of `u` and `l` is written to a file), then builds the scenarios. The first call is slow because Julia compiles the solvers.

In [ ]:
N_SCENARIOS = 48
SIGMA = 0.2

load_generator = LoadScenariosFromAggProfile(
    agg_load_name="default",
    sigma=SIGMA,
    change_reactive_power=True,
    global_range=0.4,
    max_scaling_factor=4.0,
    step_size=0.1,
    start_scaling_factor=1.0,
)

log_path = OUTPUT_DIR / "load_scenarios_demo.log"
log_path.unlink(missing_ok=True)
scenarios = load_generator(net, n_scenarios=N_SCENARIOS, scenarios_log=str(log_path), max_iter=200, seed=0)

print("\nscenarios shape (n_buses, n_scenarios, [P, Q]):", scenarios.shape)
if log_path.exists():
    print("\nLog of the OPF search:\n" + log_path.read_text())

### 3.3 Global trend and local noise

Every load is the nominal value times a **common** factor (the profile) times an **independent** random factor. We can separate the two: the mean ratio across loads recovers the global factor, and what remains is the noise, which must stay within $[1-\sigma, 1+\sigma]$ up to the approximation made by averaging.

**Try it yourself**
- Set `SIGMA = 0` in the previous cell and re-run: all loads then move together.
- Set `change_reactive_power=False` and check that `scenarios[:, :, 1]` is constant over scenarios.

In [ ]:
loaded = net.Pd > 0  # buses with zero load stay at zero whatever the factor
ratio = scenarios[loaded, :, 0] / net.Pd[loaded, None]  # (n_loads, n_scenarios)
global_factor = ratio.mean(axis=0)
local_noise = ratio / global_factor

fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].plot(global_factor)
axes[0].set(title="Global factor per scenario", xlabel="scenario", ylabel="factor vs nominal load")
axes[1].plot(scenarios[:, :, 0].sum(axis=0))
axes[1].axhline(net.Pd.sum(), ls="--", c="gray", label="nominal total")
axes[1].set(title="Total active load", xlabel="scenario", ylabel="MW")
axes[1].legend()
axes[2].hist(local_noise.ravel(), bins=30)
axes[2].axvline(1 - SIGMA, c="r", ls="--")
axes[2].axvline(1 + SIGMA, c="r", ls="--")
axes[2].set(title=f"Local noise (expected in 1 +/- {SIGMA})", xlabel="p / (p_nominal * global factor)")
plt.tight_layout()
plt.show()

---

## Part 4: Perturbations

The three other ingredients modify a **copy** of the network. Each perturbation is a *generator* (in the Python sense): it takes a network or a stream of networks and yields perturbed copies, so they can be chained lazily. None of the cells in this part solves a power flow, so they are fast.

| Perturbation | What it changes | Key parameters |
|---|---|---|
| `NMinusKGenerator` | Switches off every combination of up to `k` branches, keeping only those that leave the grid connected | `k` |
| `RandomComponentDropGenerator` | Randomly switches off up to `k` branches and/or generators | `n_topology_variants`, `k`, `elements`, `outage_count_probabilities` |
| `PermuteGenCostGenerator` | Shuffles cost coefficients between generators | none |
| `PerturbGenCostGenerator` | Multiplies cost coefficients by $\mathcal U(\max(0,1-\sigma), 1+\sigma)$ | `sigma` |
| `PerturbAdmittanceGenerator` | Multiplies each branch `R` and `X` by an independent $\mathcal U(\max(0,1-\sigma), 1+\sigma)$ | `sigma` |

Generators with constant-only costs ($c_1 = c_2 = 0$) are never permuted or scaled, and the generator at the slack bus is never dropped.

### 4.1 Topology

`outages()` below compares a variant with the base grid and lists what was switched off. The N-1 enumeration shows an important constraint: outages that would **island** part of the grid are rejected, because the solvers need a single connected component.

In [ ]:
from collections import Counter

from gridfm_datakit.perturbations.topology_perturbation import (
    NMinusKGenerator,
    RandomComponentDropGenerator,
)


def outages(variant: Network) -> dict:
    """Branches and generators in service in `net` but switched off in `variant`."""
    return {
        "branches": np.flatnonzero((net.branches[:, BR_STATUS] == 1) & (variant.branches[:, BR_STATUS] == 0)).tolist(),
        "gens": np.flatnonzero((net.gens[:, GEN_STATUS] == 1) & (variant.gens[:, GEN_STATUS] == 0)).tolist(),
    }


# Exhaustive N-1: the intact grid (N-0) plus every single branch outage that keeps the grid connected.
n_minus_1 = list(NMinusKGenerator(1, net).generate(net))
kept = {b for v in n_minus_1 for b in outages(v)["branches"]}
islanding = sorted({int(i) for i in net.idx_branches_in_service} - kept)
print(f"N-1 topologies: {len(n_minus_1)} ({len(n_minus_1) - len(kept)} intact + {len(kept)} single-branch outages)")
print(f"Branches rejected because their loss would island the grid: {islanding}")
for b in islanding:
    print(f"  branch {b}: bus {int(net.branches[b, F_BUS])} - bus {int(net.branches[b, T_BUS])}")

# Random outages with k=2: probability 10% of N-0, 80% of N-1, 10% of N-2.
np.random.seed(0)
random_drop = RandomComponentDropGenerator(
    n_topology_variants=200, k=2, base_net=net, elements=["branch", "gen"],
    outage_count_probabilities=[0.1, 0.8, 0.1],
)
variants = list(random_drop.generate(net))
sampled = [outages(v) for v in variants]
print("\nRandom sampling, outages per variant:", dict(sorted(Counter(len(o["branches"]) + len(o["gens"]) for o in sampled).items())))
print("Branch outages:", sum(len(o["branches"]) for o in sampled), "| generator outages:", sum(len(o["gens"]) for o in sampled))

`NMinusKGenerator` is exhaustive and deterministic, so it is only practical for small `k` or small grids (the number of topologies grows combinatorially). `RandomComponentDropGenerator` samples a fixed number of variants, which scales to large grids. `outage_count_probabilities` controls the mix of N-0, N-1, N-2 ... (index `i` is the probability of `i` outages). It needs a recent `gridfm-datakit` (the tutorial was written against 1.1.0; the installed version is printed in the setup cell).

### 4.2 Generator costs and admittances

Both generators take a *stream* of networks, so a helper wraps one copy of the base network. Costs only matter when an OPF is solved: they change which generators the optimizer prefers. In the output below, the cost rows of `permuted` are the same rows as `base` in another order, while `scaled` has new values. For admittances, `R` and `X` are scaled **independently**, so the `X/R` ratio of a branch changes too.

In [ ]:
from gridfm_datakit.perturbations.admittance_perturbation import PerturbAdmittanceGenerator
from gridfm_datakit.perturbations.generator_perturbation import (
    PermuteGenCostGenerator,
    PerturbGenCostGenerator,
)


def first_variant(generator) -> Network:
    return next(generator.generate(iter([net.copy_for_perturbation()])))


np.random.seed(1)
permuted = first_variant(PermuteGenCostGenerator(net))
scaled = first_variant(PerturbGenCostGenerator(net, sigma=0.5))
display(pd.concat({"base": cost_frame(net), "permuted": cost_frame(permuted), "scaled (sigma=0.5)": cost_frame(scaled)}, axis=1))

perturbed_adm = first_variant(PerturbAdmittanceGenerator(net, sigma=0.2))
r_ratio = perturbed_adm.branches[:, BR_R] / net.branches[:, BR_R]
x_ratio = perturbed_adm.branches[:, BR_X] / net.branches[:, BR_X]
print(f"R scaling factors in [{r_ratio.min():.3f}, {r_ratio.max():.3f}], X scaling factors in [{x_ratio.min():.3f}, {x_ratio.max():.3f}]")
print("R and X scaled independently:", not np.allclose(r_ratio, x_ratio))

### 4.3 Chaining perturbations

In the real pipeline the perturbations are chained as lazy generators: nothing is computed until a sample is requested, and each sample then flows through all stages. The cell below chains them in the **OPF-mode order** (topology, then costs, then admittances). In `pf` mode, costs and admittances are applied first and the topology afterwards, on the OPF result (see Part 1).

**Try it yourself**
- Replace `RandomComponentDropGenerator` by `NMinusKGenerator(1, net)` in the chain and count the samples.
- Re-run with another `np.random.seed(...)`: does the same branch always get dropped?

In [ ]:
np.random.seed(2)
pipeline = RandomComponentDropGenerator(5, 1, net, ["branch", "gen"]).generate(net)
pipeline = PermuteGenCostGenerator(net).generate(pipeline)
pipeline = PerturbAdmittanceGenerator(net, sigma=0.2).generate(pipeline)

rows = []
for i, sample in enumerate(pipeline):
    out = outages(sample)
    rows.append({
        "sample": i,
        "dropped_branches": out["branches"],
        "dropped_gens": out["gens"],
        "costs_changed": not np.allclose(sample.gencosts, net.gencosts),
        "max |R ratio - 1|": np.abs(sample.branches[:, BR_R] / net.branches[:, BR_R] - 1).max(),
    })
display(pd.DataFrame(rows))

---

## Part 5: Configuration

A whole generation run is described by one YAML file (or an equivalent Python dict). It has six blocks, one per ingredient of Part 1 plus `settings`:

| Block | Controls | Notable keys |
|---|---|---|
| `network` | Which grid | `name`, `source` (`pglib` or `file`), `network_dir` (for `file`), `reader` (`native` or `powsybl`) |
| `load` | Load scenarios | `generator`, `agg_profile`, `scenarios`, `sigma`, `global_range`, `step_size` ... |
| `topology_perturbation` | Outages | `type` (`none`, `n_minus_k`, `random`), `k`, `n_topology_variants`, `elements` |
| `generation_perturbation` | Generator costs | `type` (`none`, `cost_permutation`, `cost_perturbation`), `sigma` |
| `admittance_perturbation` | Branch R and X | `type` (`none`, `random_perturbation`), `sigma` |
| `settings` | Run options | `mode` (`pf`/`opf`), `data_dir`, `seed`, `include_dc_res`, `pf_fast`, `num_processes` |

Keys that do not apply to the chosen `type` trigger a warning rather than being silently ignored. The `seed` makes a run reproducible **only if every other parameter is identical**; leave it `null` for independent runs. The config below is deliberately small: 20 load scenarios, about 3 topology variants each.

In [ ]:
BASE_CONFIG = {
    "network": {"name": NETWORK_NAME, "source": "pglib"},
    "load": {
        "generator": "agg_load_profile",
        "agg_profile": "default",
        "scenarios": 20,
        "sigma": 0.2,
        "change_reactive_power": True,
        "global_range": 0.4,
        "max_scaling_factor": 4.0,
        "step_size": 0.1,
        "start_scaling_factor": 1.0,
    },
    "topology_perturbation": {
        "type": "random",
        "k": 1,
        "n_topology_variants": 3,
        "elements": ["branch", "gen"],
    },
    "generation_perturbation": {"type": "cost_permutation"},
    "admittance_perturbation": {"type": "random_perturbation", "sigma": 0.2},
    "settings": {
        "mode": "pf",
        "data_dir": str(OUTPUT_DIR / "pf"),
        "num_processes": 4,
        "large_chunk_size": 1000,
        "overwrite": True,
        "include_dc_res": True,
        "enable_solver_logs": False,
        "pf_fast": True,
        "dcpf_fast": True,
        "opf_formulation": "polar",
        "max_iter": 200,
        "seed": 42,
    },
}

config_path = OUTPUT_DIR / "tutorial_config.yaml"
config_path.write_text(yaml.safe_dump(BASE_CONFIG, sort_keys=False), encoding="utf-8")
print(config_path.read_text(encoding="utf-8"))

RAW_DIR = Path(BASE_CONFIG["settings"]["data_dir"]) / NETWORK_NAME / "raw"

### 5.2 From configuration to objects

`generate_power_flow_data` turns each block into the objects of Parts 3 and 4 with the helpers of `gridfm_datakit.utils.param_handler`. The cell below does the same by hand, which is a quick way to check what a config will really instantiate before running anything long.

**Try it yourself**
- Change `topology_perturbation.type` to `n_minus_k` (remove `n_topology_variants`) and see which class is built.
- Add a key that does not belong to a block (for example `sigma` under `generation_perturbation` with `type: cost_permutation`) and watch for the warning.

In [ ]:
from gridfm_datakit.utils.param_handler import (
    NestedNamespace,
    get_load_scenario_generator,
    initialize_admittance_generator,
    initialize_generation_generator,
    initialize_topology_generator,
)

args = NestedNamespace(**BASE_CONFIG)
built = {
    "load": get_load_scenario_generator(args.load),
    "topology_perturbation": initialize_topology_generator(args.topology_perturbation, net),
    "generation_perturbation": initialize_generation_generator(args.generation_perturbation, net),
    "admittance_perturbation": initialize_admittance_generator(args.admittance_perturbation, net),
}
for block, obj in built.items():
    print(f"{block:25s} -> {type(obj).__name__}")

---

## Part 6: Running a generation

`generate_power_flow_data(config)` accepts a path to a YAML file or a dict and returns the paths of the files it wrote. With the config above it:

1. loads `case14_ieee` and builds 20 load scenarios (an OPF search for `u` first),
2. for each scenario, perturbs costs and admittances, solves the AC-OPF, then creates 3 topology variants and solves an AC-PF (and DC-PF) on each,
3. writes the results in `tutorial_output/pf/case14_ieee/raw/`.

Expect up to roughly 60 samples. Some variants can fail to converge; they are skipped and logged to `error.log` (see Part 7). This cell needs Julia; the first run compiles the solvers and is the slowest step of the notebook. `overwrite: true` lets you re-run it safely.

In [ ]:
from gridfm_datakit.generate import generate_power_flow_data

file_paths = generate_power_flow_data(str(config_path))
print("\nFiles written:")
for name, path in file_paths.items() if isinstance(file_paths, dict) else enumerate(file_paths):
    print(f"  {name}: {path}")

---

## Part 7: The output

### 7.1 Files

The output folder holds one table per component plus run metadata. Tables are Parquet datasets split in `scenario_partition=<n>` sub-folders (about `large_chunk_size` samples each), which `pandas.read_parquet` reads transparently when given the folder.

| Entry | Content |
|---|---|
| `bus_data.parquet`, `gen_data.parquet`, `branch_data.parquet` | Inputs and results per bus, generator and branch |
| `y_bus_data.parquet` | Non-zero entries of the bus admittance matrix per sample |
| `runtime_data.parquet` | Solver times per sample |
| `scenarios_agg_load_profile.parquet` | The load scenarios themselves (`load_scenario` x `load`) |
| `n_scenarios.txt`, `config.yaml`, `args.log` | Number of samples and the config actually used |
| `error.log`, `tqdm.log` | Failures and progress |
| `scenarios_agg_load_profile.log`, `.html` | Log of the `u` / `l` search and an interactive plot of the load scenarios |

In [ ]:
def size_kb(path: Path) -> float:
    files = [path] if path.is_file() else [f for f in path.rglob("*") if f.is_file()]
    return sum(f.stat().st_size for f in files) / 1024


listing = pd.DataFrame(
    [{"entry": p.name + ("/" if p.is_dir() else ""), "size_kB": size_kb(p)} for p in sorted(RAW_DIR.iterdir())]
)
display(listing)

print("Partitions of bus_data.parquet:", sorted(p.name for p in (RAW_DIR / "bus_data.parquet").iterdir() if p.is_dir()))
print("n_scenarios.txt:", (RAW_DIR / "n_scenarios.txt").read_text().strip())

error_log = RAW_DIR / "error.log"
if error_log.exists():
    lines = error_log.read_text(errors="replace").splitlines()
    print(f"\nerror.log: {len(lines)} lines; last ones:")
    print("\n".join(lines[-8:]))

### 7.2 The tables

Each row is a (sample, component) pair. The columns to know:

- `scenario`: unique id of the **sample** (load scenario + topology variant + perturbations). `load_scenario_idx`: id of the **load scenario**, shared by all variants built on it.
- `bus_data`: `bus`, `Pd`, `Qd`, `Pg`, `Qg`, `Vm`, `Va`, `PQ` / `PV` / `REF` flags, `vn_kv`, limits (`min_vm_pu`, `max_vm_pu`), `GS`, `BS`, and `Va_dc`, `Pg_dc` when `include_dc_res` is true.
- `gen_data`: `idx`, `bus`, `p_mw`, `q_mvar`, `min_p_mw`, `max_p_mw`, `min_q_mvar`, `max_q_mvar`, cost coefficients `cp0_eur`, `cp1_eur_per_mw`, `cp2_eur_per_mw2`, `in_service`, `is_slack_gen`, and `p_mw_dc`.
- `branch_data`: `idx`, `from_bus`, `to_bus`, flows `pf`, `qf`, `pt`, `qt`, parameters `r`, `x`, `b`, the four admittance terms `Yff`, `Yft`, `Ytf`, `Ytt` (real and imaginary parts), `tap`, `shift`, `rate_a`, `br_status`, `ang_min`, `ang_max`, and `pf_dc`, `pt_dc`.
- `runtime_data`: `ac` and `dc` solve times in seconds.
- Units: MW, MVAr, degrees; `Vm` and the branch parameters are per unit.

The next cell loads them and shows how samples relate to load scenarios.

In [ ]:
def read_raw(raw_dir: Path) -> dict:
    names = ["bus", "gen", "branch", "y_bus", "runtime"]
    return {name: pd.read_parquet(raw_dir / f"{name}_data.parquet") for name in names}


data = read_raw(RAW_DIR)
bus, gen, branch, y_bus, runtime = (data[k] for k in ["bus", "gen", "branch", "y_bus", "runtime"])

for name, frame in data.items():
    print(f"{name:8s} {frame.shape}")

n_samples = bus["scenario"].nunique()
print(f"\n{n_samples} samples built from {bus['load_scenario_idx'].nunique()} load scenarios")
print("Samples per load scenario:", bus.groupby("load_scenario_idx")["scenario"].nunique().value_counts().to_dict(), "(count: n_scenarios)")

print("\nbus_data:")
display(bus.head(len(net.buses)))
print("branch_data (first rows):")
display(branch.head())

load_scenarios = pd.read_parquet(RAW_DIR / "scenarios_agg_load_profile.parquet")
print("scenarios_agg_load_profile:", load_scenarios.shape)
display(load_scenarios.head())

### 7.3 Physical sanity checks

A dataset is only useful if it obeys physics. Three quick checks:

1. **Active power balance** per sample: generation minus load equals the branch losses (`pf + pt` summed over branches, positive when power is dissipated) plus the shunt losses. The residual should be at solver precision.
2. **AC vs DC**: the DC approximation (`pf_dc`, `Va_dc`) should track the AC solution closely on most samples and deviate more on heavily loaded or stressed ones.
3. **Runtime**: AC solves cost more than DC solves.

In [ ]:
by_sample = bus.groupby("scenario")
losses = branch.assign(loss=branch["pf"] + branch["pt"]).groupby("scenario")["loss"].sum()
shunt = (bus["GS"] * bus["Vm"] ** 2 * net.baseMVA).groupby(bus["scenario"]).sum()
residual = by_sample["Pg"].sum() - by_sample["Pd"].sum() - losses - shunt
print(f"Active power balance residual over {len(residual)} samples: max |residual| = {residual.abs().max():.2e} MW")
print(f"Total losses: mean {losses.mean():.2f} MW ({100 * (losses / by_sample['Pd'].sum()).mean():.1f}% of demand)")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ok = branch.dropna(subset=["pf_dc"])
lim = [ok["pf"].min(), ok["pf"].max()]
axes[0].scatter(ok["pf_dc"], ok["pf"], s=6, alpha=0.4)
axes[0].plot(lim, lim, "k--")
axes[0].set(title="Branch active flow", xlabel="DC pf (MW)", ylabel="AC pf (MW)")

ok_bus = bus.dropna(subset=["Va_dc"])
lim = [ok_bus["Va"].min(), ok_bus["Va"].max()]
axes[1].scatter(ok_bus["Va_dc"], ok_bus["Va"], s=6, alpha=0.4)
axes[1].plot(lim, lim, "k--")
axes[1].set(title="Voltage angle", xlabel="DC Va (deg)", ylabel="AC Va (deg)")

runtime[["ac", "dc"]].plot.box(ax=axes[2])
axes[2].set(title="Solver runtime per sample", ylabel="seconds")
plt.tight_layout()
plt.show()

print(f"Mean |Va - Va_dc| = {(ok_bus['Va'] - ok_bus['Va_dc']).abs().mean():.3f} deg")

### 7.4 Operating-limit violations in PF mode

In `pf` mode the dispatch comes from the intact grid, so after an outage the solution may violate voltage, generator or thermal limits. These samples are valid physics but **infeasible operating points**, which is useful when training a PF solver and harmful when training an OPF solver. The function below flags them per sample (it is reused in Part 9). Tolerances: 1e-4 p.u. on voltages, 0.1% on branch loading, 1e-3 MW / MVAr on generators.

**Try it yourself**
- Which branches overload most often, and are they next to the dropped branch?
- Does the violation rate grow when you set `n_topology_variants` and `k` higher in the config?

In [ ]:
def limit_violations(bus, branch, gen, vm_tol=1e-4, flow_tol=1e-3, gen_tol=1e-3) -> pd.DataFrame:
    """One row per sample, with a boolean flag per type of operating-limit violation."""
    vm_out = (bus["Vm"] < bus["min_vm_pu"] - vm_tol) | (bus["Vm"] > bus["max_vm_pu"] + vm_tol)

    apparent = np.maximum(np.hypot(branch["pf"], branch["qf"]), np.hypot(branch["pt"], branch["qt"]))
    overload = (branch["rate_a"] > 0) & (apparent > branch["rate_a"] * (1 + flow_tol))

    on = gen["in_service"] == 1
    gen_out = on & (
        (gen["p_mw"] > gen["max_p_mw"] + gen_tol) | (gen["p_mw"] < gen["min_p_mw"] - gen_tol)
        | (gen["q_mvar"] > gen["max_q_mvar"] + gen_tol) | (gen["q_mvar"] < gen["min_q_mvar"] - gen_tol)
    )

    flags = pd.DataFrame({
        "voltage": vm_out.groupby(bus["scenario"]).any(),
        "overload": overload.groupby(branch["scenario"]).any(),
        "gen_limit": gen_out.groupby(gen["scenario"]).any(),
    })
    flags["any"] = flags.any(axis=1)
    return flags


violations = limit_violations(bus, branch, gen)
print("Share of samples violating each limit (PF mode):")
display(violations.mean().to_frame("share"))

n_outages = ((branch["br_status"] == 0).groupby(branch["scenario"]).sum()
             + (gen["in_service"] == 0).groupby(gen["scenario"]).sum()).rename("n_outages")
by_outages = violations["any"].groupby(n_outages).agg(["mean", "size"]).rename(columns={"mean": "violation_rate", "size": "samples"})
display(by_outages)

ax = by_outages["violation_rate"].plot.bar(rot=0, figsize=(5, 3))
ax.set(xlabel="number of outaged components", ylabel="share of samples with a violation", title="PF mode: violations vs outages")
plt.show()

---

## Part 8: Validation and statistics from the command line

Besides the Python API, the package installs a `gridfm_datakit` CLI with these sub-commands: `generate`, `validate`, `stats`, `plots`, `interactive` and `setup_pm`. The next cell calls the CLI from the notebook with the current interpreter (so it does not depend on `PATH`) to:

- `validate`: re-run the integrity checks (indexing, completeness, power balance at buses, branch flow recomputation, limits in `opf` mode, DC columns). It raises an error if the dataset is inconsistent, so it is a good gate before training. `--n-partitions 0` checks all partitions.
- `stats`: print and plot summary statistics.
- `plots`: draw feature distributions.

In [ ]:
CLI_ENTRY = "import sys; from gridfm_datakit.cli import main; main()"


def run_cli(*cli_args) -> int:
    """Run `gridfm_datakit <args>` in a subprocess and print its (truncated) output."""
    result = subprocess.run(
        [sys.executable, "-c", CLI_ENTRY, *map(str, cli_args)],
        capture_output=True, text=True,
    )
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print(result.stderr[-3000:])
    print(f"exit code: {result.returncode}")
    return result.returncode


run_cli("validate", RAW_DIR, "--n-partitions", 0)

In [ ]:
run_cli("stats", RAW_DIR, "--n-partitions", 0)
run_cli("plots", RAW_DIR, "--n-partitions", 0)

# stats_plot.png summarizes the whole dataset; feature_plots/ has one distribution image per bus feature.
for image_path in [RAW_DIR / "stats_plot.png", RAW_DIR / "feature_plots" / "distribution_Vm.png"]:
    if image_path.exists():
        print(image_path.relative_to(RAW_DIR))
        display(Image(filename=str(image_path)))
    else:
        print("Not found:", image_path.relative_to(RAW_DIR))

---

## Part 9: PF mode vs OPF mode

Same config, only `settings.mode` changes. In `opf` mode the topology is perturbed **before** the OPF, so every sample is a feasible, cost-optimal operating point of its own topology and the violation rate of 7.4 should drop to (almost) zero. The price is a more expensive generation, since each sample needs an AC-OPF solve.

Set `RUN_OPF_COMPARISON = False` to skip this part. It needs Julia and takes longer than Part 6, since the OPF is solved for every topology variant.

In [ ]:
RUN_OPF_COMPARISON = True

if RUN_OPF_COMPARISON:
    opf_config = copy.deepcopy(BASE_CONFIG)
    opf_config["settings"]["mode"] = "opf"
    opf_config["settings"]["data_dir"] = str(OUTPUT_DIR / "opf")
    generate_power_flow_data(opf_config)

    opf_raw_dir = Path(opf_config["settings"]["data_dir"]) / NETWORK_NAME / "raw"
    run_cli("validate", opf_raw_dir, "--n-partitions", 0)

    opf = read_raw(opf_raw_dir)
    comparison = pd.DataFrame({
        "pf mode": limit_violations(bus, branch, gen).mean(),
        "opf mode": limit_violations(opf["bus"], opf["branch"], opf["gen"]).mean(),
    })
    comparison.loc["samples"] = [bus["scenario"].nunique(), opf["bus"]["scenario"].nunique()]
    display(comparison)

---

## Summary

- A datakit sample is one solved operating point. It is diversified by load scenarios, topology outages, generator-cost changes and admittance changes.
- `Network` wraps the MATPOWER arrays, addressed through the `idx_*` constants; perturbations are lazy generators working on copies.
- Load scenarios combine a real aggregated profile scaled between `l` and `u` (found with OPF) with independent per-load noise.
- A whole run is one YAML config; `settings.mode` chooses between `pf` (stressed states, possible limit violations) and `opf` (feasible, cost-optimal states).
- Output is partitioned Parquet; `scenario` identifies a sample, `load_scenario_idx` the load scenario behind it. Always run `validate` before training.

## Troubleshooting

| Symptom | Likely cause and action |
|---|---|
| `RuntimeError: Julia / PowerModels is not available` | Run `gridfm_datakit setup_pm` and make sure Julia could download its packages. |
| First Julia call takes minutes | Normal: package compilation. Later calls reuse it. |
| Case download fails | PGLib is fetched once from the internet; check network/proxy access, or use `source: file` with a local `.m` file. |
| Fewer samples than expected | Non-converging scenarios are skipped; read `error.log`. |
| `validate` raises an `AssertionError` | The message names the failing check (balance, indexing, limits...); inspect the corresponding table. |
| Unsupported Python version | Use Python 3.10 or 3.11. |

## Going further

1. **Other grids**: set `NETWORK_NAME` to a larger PGLib case and compare runtime, violation rate and the `u` found in the log.
2. **Exhaustive topologies**: use `type: n_minus_k` with `k: 1` and compare the dataset with the random one.
3. **Costs**: switch to `type: cost_perturbation` with `sigma: 0.5` in `opf` mode and check that the dispatch (`gen_data.p_mw`) changes for the same load scenario.
4. **Load realism**: switch `load.generator` to `powergraph` and compare the distribution of `Pd` with `agg_load_profile`.
5. **Dataset quality**: for the PF dataset, plot `Vm` and `Va` distributions per bus type and decide if the coverage is enough for your use case.
6. **Next steps**: train a model on the generated data with the GridFM GraphKit tutorial, and explore the `interactive` CLI command for a UI to build configs.